In [1]:
from matplotlib.pyplot import cm
import matplotlib.pyplot as plt
import matplotlib.ticker as tkr

from tools.constants import *
from tools.metrics import *
from tools.tools_plt import *
from tools.tools_rec import *
from tools.utils import load_data

obj_keys = ['object_mse', 'object_mse_nll']

SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 14

plt.rc('font', size=MEDIUM_SIZE)         
plt.rc('axes', titlesize=BIGGER_SIZE)    
plt.rc('axes', labelsize=BIGGER_SIZE)    
plt.rc('xtick', labelsize=SMALL_SIZE)    
plt.rc('ytick', labelsize=SMALL_SIZE)    
plt.rc('legend', fontsize=MEDIUM_SIZE)  
plt.rc('figure', titlesize=BIGGER_SIZE) 
plt.rc('figure', dpi=300)

# Initial object and probe

In [ ]:
### Plot object and probe types ###
obj = simulate_object(PATH, resize=OBJ_SIZE, bc='periodic')
probe = simulate_probe(OBJ_SIZE, r_ratio=5, propagate=True, distance=60)
grad_probe = simulate_multiprobe_grad(OBJ_SIZE, r_ratio=5, r_inner=0, propagate=True, distance=60, weights=[0, 100, 0])[1]
bandlim_probe = simulate_probe(OBJ_SIZE, r_ratio=5, propagate=True, distance=60, band_lim_rand=True)

fig, ax = plt.subplots(2, 4, figsize=(16, 6))

im1 = ax[0, 0].imshow(t.abs(obj), cmap='magma')
fig.colorbar(im1, ax=ax[0, 0], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
im2 = ax[1, 0].imshow(t.angle(obj), cmap='magma')
fig.colorbar(im2, ax=ax[1, 0], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))

im3 = ax[0, 1].imshow(t.abs(probe), cmap='magma')
fig.colorbar(im3, ax=ax[0, 1], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
im4 = ax[1, 1].imshow(t.angle(probe), cmap='magma')
fig.colorbar(im4, ax=ax[1, 1], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))

im5 = ax[0, 2].imshow(t.abs(grad_probe), cmap='magma')
fig.colorbar(im3, ax=ax[0, 2], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
im6 = ax[1, 2].imshow(t.angle(grad_probe), cmap='magma')
fig.colorbar(im4, ax=ax[1, 2], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))

im5 = ax[0, 3].imshow(t.abs(bandlim_probe), cmap='magma')
fig.colorbar(im3, ax=ax[0, 3], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
im6 = ax[1, 3].imshow(t.angle(bandlim_probe), cmap='magma')
fig.colorbar(im4, ax=ax[1, 3], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))

ax[0, 0].set_ylabel('amplitude')
ax[1, 0].set_ylabel('phase')
ax[0, 0].set_title('object')
ax[0, 1].set_title('de-focused probe')
ax[0, 2].set_title('de-focused probe, y-derivative')
ax[0, 3].set_title('band-limited random probe')
for a in ax.flatten():
    a.set_xticks([]), a.set_yticks([])
fig.tight_layout()

fig.savefig('figures/initial_object_probe.png', bbox_inches='tight')

In [ ]:
### Plot several diffraction patterns ###
translations = set_scanning_grid(
    coord = (OBJ_SIZE, OBJ_SIZE),
    n_steps = OBJ_SIZE,
    steps_size = 1,
    add_noise = False
)
diff10, _ = get_diffractions_fluence(probe, obj, translations, 10, 'poisson')
diff100, _ = get_diffractions_fluence(probe, obj, translations, 100, 'poisson')
diff1000, _ = get_diffractions_fluence(probe, obj, translations, 1000, 'poisson')

fig, ax = plt.subplots(1, 3, figsize=(12, 3))

im0 = ax[0].imshow(t.log10(diff10[1500]), cmap='magma')
fig.colorbar(im0, ax=ax[0], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
ax[0].set_title("10 photons/pixel")

im1 = ax[1].imshow(t.log10(diff100[1500]), cmap='magma')
fig.colorbar(im1, ax=ax[1], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
ax[1].set_title("100 photons/pixel")

im2 = ax[2].imshow(t.log10(diff1000[1500]), cmap='magma')
fig.colorbar(im2, ax=ax[2], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
ax[2].set_title("1000 photons/pixel")

for a in ax.flatten():
    a.set_xticks([]), a.set_yticks([])
fig.tight_layout()

fig.savefig('figures/initial_diffractions.png', bbox_inches='tight')

# Fluence

In [ ]:
### Plot error metrics for varying fluence reconstructions ###
rec_dict = load_data('outputs/rec_fluence.pkl')

fluences = np.array(list(rec_dict['object_mse'].keys()))
mse_dict = {key: get_MSE_flu(TRUE_IMG, rec_dict[key]) for key in obj_keys}
ssim_dict = {key: get_SSIM_flu(TRUE_IMG, rec_dict[key]) for key in obj_keys}

fig = plot_metric_flu(fluences, mse_dict, ssim_dict)

fig.savefig('figures/metrics_flu.png')

In [ ]:
### Plot examples of reconstructions for MSE and MSE-PNLL optimizations, at a couple of chosen fluences ###
flu_idx = [6, 16, 24, 40, 48] # index of the chosen fluences to pass in the fluences array

fig, ax = plt.subplots(4, 5, figsize=(20, 12))
for i, flu in enumerate(flu_idx):
    
    im_abs = ax[0, i].imshow(t.abs(rec_dict['object_mse'][fluences[flu]]), cmap='magma')
    fig.colorbar(im_abs, ax=ax[0, i], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
    
    im_phase = ax[1, i].imshow(t.angle(rec_dict['object_mse'][fluences[flu]]), cmap='magma')
    fig.colorbar(im_phase, ax=ax[1, i], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
    
    im_abs = ax[2, i].imshow(t.abs(rec_dict['object_mse_nll'][fluences[flu]]), cmap='magma')
    fig.colorbar(im_abs, ax=ax[2, i], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
    
    im_phase = ax[3, i].imshow(t.angle(rec_dict['object_mse_nll'][fluences[flu]]), cmap='magma')
    fig.colorbar(im_phase, ax=ax[3, i], orientation='vertical', format=tkr.FormatStrFormatter('%.2f'))
    
    ax[0, i].set_title(f'{fluences[flu]:.1f} photons/pixel')

for a in ax.flatten():
    a.set_xticks([])
    a.set_yticks([])
ax[0, 0].set_ylabel('MSE, amplitude')
ax[1, 0].set_ylabel('MSE, phase')
ax[2, 0].set_ylabel('MSE-PNLL, amplitude')
ax[3, 0].set_ylabel('MSE-PNLL, phase')
fig.tight_layout()

fig.savefig('figures/rec_imgs.png', bbox_inches='tight')

# Steps sizes

In [ ]:
### Plot error metrics for varying step sizes reconstructions ###
rec_dict = {
    'object_mse': {},
    'loss_mse': {},
    'object_mse_nll': {},
    'loss_mse_nll': {}
}
for i in range(1, 26):
    loaded_dict = load_data(f'outputs/steps_bandlim5/rec_steps_bandlim5_{i}.pkl') # use de-focused ring probe
    for key, value in loaded_dict.items():
        rec_dict[key][i] = value

steps_sizes = np.array(list(rec_dict['object_mse'].keys()))
fluences = np.array(list(rec_dict['object_mse'][1].keys()))
mse_dict = {key: get_MSE_param_flu(TRUE_IMG, rec_dict[key]) for key in obj_keys}
ssim_dict = {key: get_SSIM_param_flu(TRUE_IMG, rec_dict[key]) for key in obj_keys}

fig = plot_metrics_params_flu(fluences, mse_dict, ssim_dict, 'steps', steps_sizes, np.arange(0, 22))
 
fig.savefig('figures/metrics_steps.png', bbox_inches='tight')

In [ ]:
### Plot mean square error evolution over varying step sizes for given fluences ###
f_idx = 14
steps_max = 21
s = 25

fig, ax = plt.subplots(1, 3, figsize=(15, 6))

ax[0].scatter(steps_sizes[:steps_max], mse_dict['object_mse'][:steps_max, f_idx], label='MSE loss', s=s, c='b')
ax[0].scatter(steps_sizes[:steps_max], mse_dict['object_mse_nll'][:steps_max, f_idx], label='MSE - PNLL losses', s=s, c='r')
ax[0].legend()

blues, reds = cm.Blues(np.linspace(0, 1, 7)), cm.Reds(np.linspace(0, 1, 7))
for f, b, r in zip(range(10,15), blues[1:], reds[1:]):
    ax[1].scatter(steps_sizes[:steps_max], mse_dict['object_mse'][:steps_max, f], s=s, c=b)
    ax[1].text(5.5, mse_dict['object_mse'][6, f]*1.2, f'$f={round(fluences[f])}$ photons/pixel')
    ax[2].scatter(steps_sizes[:steps_max], mse_dict['object_mse_nll'][:steps_max, f], s=s, c=r)
    ax[2].text(0.5, mse_dict['object_mse_nll'][1, f]*1.4, f'$f={round(fluences[f])}$ photons/pixel')

for a in ax:
    a.legend(), a.grid(), a.set_yscale('log')
    a.set_xlabel('step size')
ax[0].set_ylabel('mse')
ax[0].set_title(f'mse at $f={round(fluences[f_idx])}$ photons/pixel')
ax[1].set_title('MSE')
ax[2].set_title('MSE-PNLL')
fig.tight_layout()

fig.savefig('figures/metrics_steps_comparison.png', bbox_inches='tight')

# Multiprobe - grad

In [ ]:
### Plot error metrics for varying weighting of probe + y-directional derivative multi-mode mixture ###
weights = np.linspace([100, 0, 0], [50, 50, 0], 6)
pkl_suf = [
    np.array2string(w, separator=',').replace(' ', '').replace('.', '')
    for w in weights
]
rec_dict = {
    'object_mse': {},
    'loss_mse': {},
    'object_mse_nll': {},
    'loss_mse_nll': {}
}
for suf in pkl_suf:
    loaded_dict = load_data(f'outputs/grad_1direction/rec_grad_1direction_{suf}.pkl')
    for key, value in loaded_dict.items():
        rec_dict[key][suf] = value

modes_config = np.array(list(rec_dict['object_mse'].keys()))
fluences = np.array(list(rec_dict['object_mse'][modes_config[0]].keys()))
mse_dict = {key: get_MSE_param_flu(TRUE_IMG, rec_dict[key]) for key in obj_keys}
ssim_dict = {key: get_SSIM_param_flu(TRUE_IMG, rec_dict[key]) for key in obj_keys}

fig = plot_metrics_params_flu(fluences, mse_dict, ssim_dict, 'grad', modes_config)

fig.savefig('figures/metrics_grad_1direction.png', bbox_inches='tight')

In [ ]:
### Plot mean square error evolution over varying weighting of probe + y-directional derivative multi-mode mixture ###
f_idx = 14
s = 25

fig, ax = plt.subplots(1, 3, figsize=(15, 6))

ax[0].scatter(range(6), mse_dict['object_mse'][:, f_idx], label='MSE loss', s=s, c='b')
ax[0].scatter(range(6), mse_dict['object_mse_nll'][:, f_idx], label='MSE - PNLL losses', s=s, c='r')
ax[0].legend()

blues, reds = cm.Blues(np.linspace(0, 1, 8)), cm.Reds(np.linspace(0, 1, 8))
for f, b, r in zip(range(10,15), blues[2:], reds[2:]):
    ax[1].scatter(range(6), mse_dict['object_mse'][:, f], s=s, c=b)
    ax[1].text(0, mse_dict['object_mse'][0, f]*1.2, f'$f={round(fluences[f])}$ photons/pixel')
    ax[2].scatter(range(6), mse_dict['object_mse_nll'][:, f], s=s, c=r)
    ax[2].text(0, mse_dict['object_mse_nll'][0, f]*1.2, f'$f={round(fluences[f])}$ photons/pixel')

for a in ax:
    a.grid(), a.set_yscale('log')
    a.set_xticks(range(6), [m[:6] + ']' for m in modes_config])
    a.set_xlabel('weights configuration')
ax[0].set_ylabel('mse')
ax[0].set_title(f'mse at $f={round(fluences[f_idx])}$ photons/pixel')
ax[1].set_title('MSE')
ax[2].set_title('MSE-PNLL')
fig.tight_layout()

fig.savefig('figures/metrics_grad_1direction_comparison.png', bbox_inches='tight')

# Multiprobe - band limited random

In [ ]:
### Plot error metrics for varying increasing amount of band-limited random modes ###
rec_dict = {
    'object_mse': {},
    'loss_mse': {},
    'object_mse_nll': {},
    'loss_mse_nll': {}
}
for i in range(1, 11):
    loaded_dict = load_data(f'outputs/bandlim/rec_bandlim_{i}.pkl')
    for key, value in loaded_dict.items():
        rec_dict[key][i] = value

n_modes = np.array(list(rec_dict['object_mse'].keys()))
fluences = np.array(list(rec_dict['object_mse'][n_modes[0]].keys()))
mse_dict = {key: get_MSE_param_flu(TRUE_IMG, rec_dict[key]) for key in obj_keys}
ssim_dict = {key: get_SSIM_param_flu(TRUE_IMG, rec_dict[key]) for key in obj_keys}

f = plot_metrics_params_flu(fluences, mse_dict, ssim_dict, 'band_lim', n_modes)

f.savefig('figures/metrics_bandlim.png', bbox_inches='tight')

In [ ]:
### Plot mean square error evolution over varying weighting of probe + y-directional derivative multi-mode mixture ###
f_idx = 14
s = 25

fig, ax = plt.subplots(1, 3, figsize=(15, 6))

ax[0].scatter(n_modes, mse_dict['object_mse'][:, f_idx], label='MSE loss', s=s, c='b')
ax[0].scatter(n_modes, mse_dict['object_mse_nll'][:, f_idx], label='MSE - PNLL losses', s=s, c='r')
ax[0].legend()

blues, reds = cm.Blues(np.linspace(0, 1, 8)), cm.Reds(np.linspace(0, 1, 8))
for f, b, r in zip(range(10,15), blues[2:], reds[2:]):
    ax[1].scatter(n_modes, mse_dict['object_mse'][:, f], label=f'$f={fluences[f]:.1f}$ photons/pixel', s=s, c=b)
    ax[1].text(2, mse_dict['object_mse'][2, f], f'$f={round(fluences[f])}$ photons/pixel', rotation=20)
    ax[2].scatter(n_modes, mse_dict['object_mse_nll'][:, f], label=f'$f={fluences[f]:.1f}$ photons/pixel', s=s, c=r)
    ax[2].text(3, mse_dict['object_mse_nll'][3, f], f'$f={round(fluences[f])}$ photons/pixel', rotation=20)

for a in ax:
    a.grid(), a.set_yscale('log')
    a.set_xlabel(r'$n_{modes}$')
ax[0].set_ylabel('mse')

ax[0].set_title(f'mse at $f={fluences[f_idx]:.1f}$ photons/pixel')
ax[1].set_title('MSE')
ax[2].set_title('MSE-PNLL')
fig.tight_layout()

fig.savefig('figures/metrics_bandlim_comparison.png', bbox_inches='tight')